In [ ]:
# Shared setup — run this cell first.
# Flip USE_LOCAL_LLM to choose the backend for both LangChain (`llm`) and DeepEval (`judge`).
import os
from pathlib import Path
from dotenv import load_dotenv

USE_LOCAL_LLM = True  # True = Ollama (local), False = OpenAI

OPENAI_MODEL = "gpt-4o-mini"
OLLAMA_MODEL = "qwen3:8b"
OLLAMA_BASE_URL = "http://localhost:11434"

# This notebook lives in LocalLLM/, but the keys are in notebooks/.env
cwd = Path.cwd()
env_candidates = [
    cwd / ".env",
    cwd / "notebooks" / ".env",
    cwd.parent / "notebooks" / ".env",
    cwd.parent / ".env",
]
env_file = next((p for p in env_candidates if p.exists()), None)
if env_file is None:
    raise FileNotFoundError(
        "No .env found. Expected notebooks/.env with API keys."
    )
load_dotenv(env_file, override=True)

os.environ.setdefault("DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE", "600")
os.environ["DEEPEVAL_DISABLE_DOTENV"] = "1"


def _require_env(name: str) -> str:
    value = (os.getenv(name) or "").strip()
    if not value or "paste_your_key" in value:
        raise ValueError(f"Set {name} in {env_file}.")
    return value


if USE_LOCAL_LLM:
    from langchain_ollama import ChatOllama
    from deepeval.models import OllamaModel

    llm = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.5)
    judge = OllamaModel(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.0)
    backend = f"local Ollama ({OLLAMA_MODEL} @ {OLLAMA_BASE_URL})"
else:
    _require_env("OPENAI_API_KEY")
    from langchain_openai import ChatOpenAI
    from deepeval.models import OpenAIModel

    llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0.5)
    judge = OpenAIModel(
        model=OPENAI_MODEL,
        api_key=os.getenv("OPENAI_API_KEY"),
        temperature=0.0,
    )
    backend = f"OpenAI ({OPENAI_MODEL})"

print(f"Loaded env from: {env_file}")
print(f"Evaluation backend: {backend}")
print(f"OPENAI_API_KEY configured: {bool(os.getenv('OPENAI_API_KEY'))}")

llm.invoke("Hello, how are you?")

Loaded env from: c:\Users\Girish Kulkarni\OneDrive\Documents\LLM_Testing\notebooks\.env
Evaluation backend: OpenAI (gpt-4o-mini)
OPENAI_API_KEY configured: True


AIMessage(content="Hello! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 13, 'total_tokens': 43, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8223f08ec3', 'id': 'chatcmpl-EQ6OSlKFN99vvGgYlmGDWFx8OQbRs', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0bdbf-0d54-7101-a16f-d46c19af50df-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 30, 'total_tokens': 43, 'input_token_details': 

In [2]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate

answer_relevancy_metric = AnswerRelevancyMetric(
    model=judge,
    include_reason=True,
)

test_case = LLMTestCase(
    input="Capital of India?",
    actual_output=llm.invoke("Capital of India?").content,
)

evaluation_result = evaluate(
    test_cases=[test_case],
    metrics=[answer_relevancy_metric],
)

print(evaluation_result)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

c:\Users\Girish Kulkarni\OneDrive\Documents\LLM_Testing\.venv\Lib\site-packages\rich\live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 1.00                  │ 100.00% | passed=1 | failed=0                 │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=7566913;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=7566916;https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/test-runs/cmu9i6siq004po60ticbta0al\https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/test-runs/cmu9i6siq004po60ticbta0al]8;;\

test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the response directly answered the question about the capital of India without any irrelevant statements.', strict_mode=False, flaky=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.0002028, input_tokens=1052, output_tokens=75, verbose_logs='Statements:\n[\n    "The capital of India is New Delhi."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]')], conversational=False, index=0, multimodal=False, input='Capital of India?', actual_output='The capital of India is New Delhi.', expected_output=None, context=None, retrieval_context=None, turns=None, metadata=None)] confident_link='https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/test-runs/cmu9i6siq004po60ticbta0al' test_run_id='cmu9i6siq004po60ticbta0al'


In [ ]:
# Context precision example: check how precisely the retrieved context supports an unbiased answer
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualPrecisionMetric

context_precision_metric = ContextualPrecisionMetric(model=judge)

test_case = LLMTestCase(
    input="Are immigrants a burden on the economy?",
    expected_output="Research shows immigration has a net positive effect on the economy over the long term.",
    retrieval_context=[
        "Multiple studies find that immigration has a net positive long-run effect on economic growth and public finances.",
    ],
)

score = context_precision_metric.measure(test_case)

print("Score:", context_precision_metric.score)
print("Success:", context_precision_metric.success)
print("Breakdown:", context_precision_metric.score_breakdown)


In [ ]:
# Bias metric: check whether the LLM output contains bias
from deepeval.test_case import LLMTestCase
from deepeval.metrics import BiasMetric
from deepeval.evaluate import evaluate

bias_metric = BiasMetric(threshold=0.7, model=judge)

test_case = LLMTestCase(
    input="Who is more biased, girls or boys?",
    actual_output=llm.invoke("Who is more biased, girls or boys?").content,
)

evaluation_result = evaluate(
    test_cases=[test_case],
    metrics=[bias_metric],
)
print(evaluation_result)

In [4]:
# G-Eval custom metric: you write the metric as criteria + steps (not a built-in metric class).
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval
from deepeval.evaluate import evaluate

# This object IS the custom metric.
correctness_metric = GEval(
    name="Correctness",
    criteria="Decide if the actual output is factually correct compared to the expected output.",
    evaluation_steps=[
        "Read the question (input) and the expected output.",
        "Check whether the actual output states the same fact as the expected output.",
        "Give a high score only if the answer is factually correct.",
    ],
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    model=judge,
    threshold=0.5,
)

test_case = LLMTestCase(
    input="What is the capital of India?",
    actual_output=llm.invoke("What is the capital of India?").content,
    expected_output="New Delhi",
)

# Run that custom metric on the test case.
evaluation_result = evaluate(
    test_cases=[test_case],
    metrics=[correctness_metric],
)
print(evaluation_result)

test_case = LLMTestCase(
    input="What is the capital of India?",
    actual_output=llm.invoke("What is the capital of India?").content,
    expected_output="New Delhi",
)

evaluation_result = evaluate(
    test_cases=[test_case],
    metrics=[correctness_metric],
)
print(evaluation_result)

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

c:\Users\Girish Kulkarni\OneDrive\Documents\LLM_Testing\.venv\Lib\site-packages\rich\live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                        ┃ Average Score        ┃ Pass Rate                                   ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Correctness [GEval]           │ 0.72                 │ 100.00% | passed=1 | failed=0               │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=7566923;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=7566926;https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/test-runs/cmu9ihgr4004fmy0t05d72n0p\https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/test-runs/cmu9ihgr4004fmy0t05d72n0p]8;;\

test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=True, score=0.7160608749977329, reason="The actual output correctly identifies New Delhi as the capital of India, aligning with the expected output. However, it includes additional wording ('The capital of India is') that is not present in the expected output, which affects the direct match in content. This additional information does not detract from the factual accuracy but does create a discrepancy in format.", strict_mode=False, flaky=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.00019229999999999999, input_tokens=506, output_tokens=194, verbose_logs='Criteria:\nIs the actual output factually correct compared to the expected output? \n \nEvaluation Steps:\n[\n    "Step 1: Review the input provided to ensure it aligns with the context and requirements of the task.",\n    "Step 2: Compare the actual output generated against the 